# GenePromoter -- DNABERT-2 fine-tuning (Colab / GPU)

Runs Phase 4 (training) and Phase 5 (evaluation) from `dl_roadmap.md` on a Colab GPU runtime.
Data (`data/raw_data.csv`, `data/{train,val,test}_1000bp.csv`) is already committed to the repo, so this notebook does not re-fetch anything.

**Before running:** Runtime -> Change runtime type -> T4 GPU.

## 1. Clone repo

In [ ]:
!git clone https://github.com/ekta120405/GenePromoter
%cd GenePromoter

## 2. Install dependencies

Do **not** reinstall `torch` -- Colab's preinstalled build already matches its CUDA runtime.

We also deliberately do **not** pin `transformers`/`tokenizers` to the versions in `requirements.txt` here. Those versions were pinned to dodge a *local Windows* problem (no Rust toolchain, no prebuilt wheel for the older `transformers==4.29.2`). Colab's image is different and actively maintained -- the latest `transformers` has a prebuilt `tokenizers` wheel, so installing unpinned avoids a from-source build entirely. Cell 4 below (the sanity check) verifies the DNABERT-2 remote-code patch still works against whatever version this resolves to, before any GPU time is spent training.

In [ ]:
!pip install -q -U transformers einops accelerate scikit-learn pandas numpy datasets requests

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (check Runtime > Change runtime type)')

## 3. Sanity check (Phase 0 / Phase 3 -- optional but fast)

Confirms the DNABERT-2 remote-code patch and tokenizer work on this runtime before committing to a full training run.

In [ ]:
!python src/check_tokenization.py data/train_1000bp.csv

## 4. Train (Phase 4) -- full-scale, GPU-accelerated

Uses the full training set and 5 epochs (vs. the 2000-sample/1-epoch CPU default). `train.py` already picks up `cuda` automatically via `torch.cuda.is_available()`.

In [ ]:
!python src/train.py --seq-len 1000 --max-train-samples -1 --epochs 5

## 5. Evaluate on held-out test set (Phase 5)

In [ ]:
!mkdir -p results
!python src/evaluate.py --seq-len 1000 --output results/test_results.txt

## 6. Bring the checkpoint back

Zip `checkpoints/best_model/` and download it (or copy to Drive) so `predict.py` and the Cloud Computing handoff can use it locally.

In [ ]:
!zip -r best_model.zip checkpoints/best_model
from google.colab import files
files.download('best_model.zip')

Alternative: mount Drive and copy instead of downloading a zip, if the checkpoint is large:
```python
from google.colab import drive
drive.mount('/content/drive')
!cp -r checkpoints/best_model /content/drive/MyDrive/GenePromoter_checkpoint
```

## 7. Improvement experiments -- lower LR + frozen-base comparison

Baseline (full fine-tune, lr=3e-5, 5 epochs): test accuracy 0.68, macro F1 0.67, checkpoint at `checkpoints/best_model`.

Two things worth trying, both cheap (same script, different flags), saved to separate output dirs so the baseline checkpoint isn't touched:

1. **Lower learning rate.** Train loss fell fast (0.66->0.43 over 4 epochs) while val F1 peaked at epoch 1 and never recovered -- a classic sign the LR is a bit high and the model is memorizing rather than generalizing. `--lr 1e-5` is a standard "try 3x lower" step.
2. **Frozen base (`--freeze-base`).** Only trains the classification head, keeping DNABERT-2's pretrained weights fixed. Fewer trainable params can mean less overfitting (even if the ceiling is lower) -- and this is also the frozen-vs-fine-tuned stretch-goal comparison from the project's research scope, not just a tuning trick.

### 7a. Experiment 1 -- lower learning rate (full fine-tune, lr=1e-5)

In [ ]:
!python src/train.py --seq-len 1000 --max-train-samples -1 --epochs 5 --lr 1e-5 --output-dir checkpoints/lr1e5_model

In [ ]:
!python src/evaluate.py --seq-len 1000 --checkpoint checkpoints/lr1e5_model --output results/test_results_lr1e5.txt

### 7b. Experiment 2 -- frozen base (only the classification head trains)

In [ ]:
!python src/train.py --seq-len 1000 --max-train-samples -1 --epochs 5 --freeze-base --output-dir checkpoints/frozen_model

In [ ]:
!python src/evaluate.py --seq-len 1000 --checkpoint checkpoints/frozen_model --output results/test_results_frozen.txt

### 7c. Compare all three results

Baseline result is already known (accuracy 0.68, macro F1 0.67, printed earlier in this notebook / already saved to your local repo). This just prints the two new ones side by side for comparison.

In [ ]:
print("=== lr=1e-5 (full fine-tune) ===")
!cat results/test_results_lr1e5.txt
print()
print("=== frozen-base ===")
!cat results/test_results_frozen.txt

### 7d. Bring the comparison home

The two results `.txt` files are tiny -- download both regardless of which wins, they're what goes in the paper's comparison table. The checkpoints are ~450MB each; only bother downloading whichever one (if any) beats the 0.68/0.67 baseline -- update the `checkpoints/...` path below to whichever experiment you want to keep.

In [ ]:
from google.colab import files
files.download('results/test_results_lr1e5.txt')
files.download('results/test_results_frozen.txt')

# If one of the experiments won, uncomment and set the winning checkpoint dir:
# !zip -r winning_model.zip checkpoints/lr1e5_model   # or checkpoints/frozen_model
# files.download('winning_model.zip')